# Tutorial 03 — Interpretability evaluation

Evaluates irAEGIS attributions along three axes:

1. **Fidelity** — necessity of top-K features by occlusion (patient-level AUC drop).
2. **Class contrastivity** — independence of Yes-/No-driving top-pathway sets.
3. **Ground-truth recovery** — recall@5 / precision@30 against injected signal on simulated cohorts.

All three metrics consume `cell_pathway_attribution.csv` and `cell_gene_attribution.csv`, written by `explainability/cell_explain.py`.


In [1]:
import os, warnings, subprocess
from pathlib import Path
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')

REPO   = Path('..').resolve()
COHORT = 'GSE189125_pre_ici'
PY     = 'python'


## Cell-level attribution

Computes per-(cell type, pathway) attribution as `M_sign × W_mask` and projects it to gene-level scores. This step is a prerequisite for all three downstream metrics.


In [2]:
_ = subprocess.run([PY, 'explainability/cell_explain.py', '--cohort', COHORT],
                   cwd=REPO, check=True)



  Cell-Level Explainability — GSE189125_pre_ici
  Output → results/iraegis/GSE189125_pre_ici/cell_explainability
  AE checkpoint: ae_encoder.pt
  CT align: 28159 / 28346 cells matched.

  [Attribution] Computing h_diff and w_CT per (CT, pathway) ...

  [Genes] Projecting pathway attributions to genes ...
    450 rows

  Done → results/iraegis/GSE189125_pre_ici/cell_explainability


## Fidelity — necessity by occlusion

Patient-level AUC drop after masking (i) the top-K=20 genes per pathway within each cell type, and (ii) every gene in each cell type whose per-CT classifier has AUC > 0.5. Larger drops indicate the attribution identifies features the classifier genuinely depends on.


In [3]:
_ = subprocess.run([PY, 'explainability/fidelity_occlusion.py', '--cohorts', COHORT],
                   cwd=REPO, check=True)

import pandas as pd
df = pd.read_csv(REPO / 'results' / 'fidelity_occlusion.csv')
df = df.rename(columns={
    'drop_pw_genes_necc':     'dAUC_pathway_genes',
    'drop_cts_compound_necc': 'dAUC_celltype_genes',
})
df[['cohort', 'n_patients', 'n_cts_masked', 'auc_full',
    'dAUC_pathway_genes', 'dAUC_celltype_genes']]



[GSE189125_pre_ici]
[relabel_by_grade] GSE189125_pre_ici: grade>=3 → 18,881 Yes cells, 10,745 No cells (16 patients)
[cell QC] Dropping 1,280/29,626 cells with <200 genes
  full                  AUC=0.9524  AUPRC=0.9627
  -genes in top-20 PWs (necc, all CTs) AUC=0.2063  Δ=-0.7460  AUPRC=0.4429  Δ=-0.5198
  -CTs with AUC>0.5 (3 CTs, 4696 cells, necc) AUC=0.3333  Δ=-0.6190  AUPRC=0.5479  Δ=-0.4148

Wrote 1 rows → results/fidelity_occlusion.csv


,cohort,n_patients,n_cts_masked,auc_full,dAUC_pathway_genes,dAUC_celltype_genes
0,GSE189125_pre_ici,16,3,0.9524,0.746,0.619


## Class contrastivity

Reports `1 − Jaccard` between the Yes-driving and No-driving top-pathway sets. A value near 1 means the two sets are nearly disjoint, i.e., the model's Yes-side and No-side explanations are class-specific.


In [4]:
_ = subprocess.run([PY, 'explainability/class_contrastivity.py'], cwd=REPO, check=True)

import pandas as pd
df = pd.read_csv(REPO / 'results' / 'explainability_validation.csv')
df['class_contrastivity'] = 1 - df['yes_no_pw_jaccard']
df[['cohort', 'pw_yes_driven', 'pw_no_driven',
    'yes_no_pw_jaccard', 'class_contrastivity']]



Wrote 10 rows → results/explainability_validation.csv


,cohort,pw_yes_driven,pw_no_driven,yes_no_pw_jaccard,class_contrastivity
0,DS_cohort1,8,7,0.077,0.923
1,DS_cohort2,11,9,0.053,0.947
2,DS_cohort3,13,6,0.056,0.944
3,GSE189125_pre_ici,59,14,0.154,0.846
4,GSE216329_integrated_pre_ici,44,27,0.278,0.722
5,GSE249898_integrated_pre_ici,10,61,0.226,0.774
6,GSE285888_pre_ici,56,34,0.389,0.611
7,RS_cohort1,11,14,0.150,0.850
8,RS_cohort2,16,19,0.111,0.889
9,RS_cohort3,12,13,0.095,0.905


## Ground-truth recovery

Pathway recall@5 and gene precision@30 against the injected ground-truth on the six simulated cohorts (`RS_*`, `DS_*`).


In [5]:
_ = subprocess.run([PY, 'explainability/ground_truth_recovery.py'], cwd=REPO, check=True)


  RS_cohort1    recall@5=1.0000  precision@30=0.9583  (n_signal_cts=4)
  RS_cohort2    recall@5=0.8750  precision@30=0.9167  (n_signal_cts=4)
  RS_cohort3    recall@5=0.8750  precision@30=1.0000  (n_signal_cts=4)
  DS_cohort1    recall@5=1.0000  precision@30=0.9917  (n_signal_cts=4)
  DS_cohort2    recall@5=0.8750  precision@30=0.9500  (n_signal_cts=4)
  DS_cohort3    recall@5=0.8750  precision@30=0.9917  (n_signal_cts=4)

Saved → results/ground_truth_recovery.csv


### Summary

| Metric | Definition | Output file |
|---|---|---|
| Fidelity ΔAUC | AUC drop when top-K attribution features are masked | `results/fidelity_occlusion.csv` |
| Class contrastivity | `1 − Jaccard(top-Yes, top-No)` over pathways | `results/explainability_validation.csv` |
| Pathway recall@5 / gene precision@30 | Recovery of injected ground-truth on simulations | `results/ground_truth_recovery.csv` |
